# Model Gempa M>=5 Bulanan (Radius 50km + Region)
Notebook ini adalah versi step-by-step dari `model_m5_monthly_radius50.py`.

Tujuan:
- Memudahkan eksplorasi per langkah
- Menjalankan training dan evaluasi per mode (spatial/region)
- Menjalankan inferensi 1 bulan ke depan

In [ ]:
from pathlib import Path
import json
import pandas as pd

from model_m5_monthly_radius50 import (
    DATA_PATH,
    ARTIFACT_DIR,
    DatasetConfig,
    load_filtered_events,
    build_spatial_panel,
    build_region_panel,
    _build_feature_columns,
    _split_train_test_by_month,
    _evaluate_panel,
    train_and_save,
    predict_next_month_point,
    predict_next_month_region,
)
import joblib

print('DATA_PATH:', DATA_PATH)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

## 1) Load dan cek data event M>=5

In [ ]:
events = load_filtered_events(DATA_PATH)
print('Filtered events:', len(events))
print('Month min:', events['month'].min())
print('Month max:', events['month'].max())
events.head()

## 2) Bentuk panel spatial dan region

In [ ]:
cfg = DatasetConfig()

spatial_panel, cells = build_spatial_panel(events, cfg)
region_panel, region_map = build_region_panel(events, cfg)

print('Spatial panel shape:', spatial_panel.shape)
print('Unique cells:', len(cells))
print('Region panel shape:', region_panel.shape)
print('Unique regions:', len(region_map))

display(cells.head())
display(region_map.head())

## 3) Evaluasi mode spatial (MAE dan RMSE)

In [ ]:
spatial_features = _build_feature_columns(spatial_panel, extra_cols=['lat_bin', 'lon_bin'])
sp_train, sp_test = _split_train_test_by_month(spatial_panel, cfg.test_horizon)
spatial_metrics, spatial_pred_test = _evaluate_panel(
    sp_train,
    sp_test,
    feature_cols=spatial_features,
    group_cols=['entity', 'lat_bin', 'lon_bin'],
)
print(json.dumps(spatial_metrics, indent=2))
spatial_pred_test.head()

## 4) Evaluasi mode region (MAE dan RMSE)

In [ ]:
region_features = _build_feature_columns(region_panel, extra_cols=['region_code'])
rg_train, rg_test = _split_train_test_by_month(region_panel, cfg.test_horizon)
region_metrics, region_pred_test = _evaluate_panel(
    rg_train,
    rg_test,
    feature_cols=region_features,
    group_cols=['entity', 'region_code'],
)
print(json.dumps(region_metrics, indent=2))
region_pred_test.head()

## 5) Train full model dan simpan artifact

In [ ]:
summary = train_and_save(data_path=DATA_PATH, artifact_dir=ARTIFACT_DIR)
print(json.dumps(summary, indent=2))

## 6) Load model bundle untuk inferensi

In [ ]:
bundle_path = ARTIFACT_DIR / 'saved_models' / 'm5_monthly_radius50_bundle.joblib'
bundle = joblib.load(bundle_path)
print('Loaded:', bundle_path)
print(json.dumps(bundle['metadata'], indent=2))

## 7) Inferensi mode titik (proxy radius 50km)

In [ ]:
res_point = predict_next_month_point(bundle, lat=-6.2, lon=106.8)
print(json.dumps(res_point, indent=2))

## 8) Inferensi mode region

In [ ]:
available_regions = sorted(bundle['region']['region_map']['entity'].unique().tolist())
print('Total regions:', len(available_regions))
print('Sample regions:', available_regions[:10])

res_region = predict_next_month_region(bundle, region_name='Banda Sea')
print(json.dumps(res_region, indent=2))